
**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

<strong> **Day 5 — Error handling, custom exceptions, and defensive programming**</strong>

**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

<strong> **Concept 1: what an <u> exception</u> actually is**</strong>


<h6>
<strong> The analogy </strong>

A restaurant kitchen has a rule: if an order can’t be made, the waiter doesn’t guess and bring something random — they come back and tell you exactly what went wrong. “We’re out of that ingredient.” “That dish doesn’t exist.” The kitchen raises a specific problem back to whoever asked.

Python works the same way. When something goes wrong inside a function, instead of returning garbage or silently continuing, Python raises an exception — a structured signal that says exactly what went wrong and where.

BaseException

├── Exception

│   ├── ValueError        — wrong value for the type

│   ├── TypeError         — wrong type

│   ├── KeyError          — missing dict key

│   ├── IndexError        — list index out of range

│   ├── FileNotFoundError — file doesn't exist

│   ├── AttributeError    — object has no such attribute

│   ├── ImportError       — can't import module

│   ├── JSONDecodeError   — malformed JSON

│   ├── ZeroDivisionError — division by zero

│   └── ...

└── KeyboardInterrupt     — Ctrl+C

When Python raises an exception, it does two things:

1, Creates an instance of the exception class, with a message.

2, "Throws" it up the call stack — the current function stops executing, and the exception propagates to the caller. If nobody catches it, the program crashes and prints a traceback.
</h6>

In [1]:
# An exception is just an object — an instance of a class
# This is all a ValueError actually is:
e = ValueError("null_count must be >= 0, got -5")

print(type(e))       # <class 'ValueError'>
print(e.args)        # ('null_count must be >= 0, got -5',)
print(str(e))        # null_count must be >= 0, got -5

<class 'ValueError'>
('null_count must be >= 0, got -5',)
null_count must be >= 0, got -5


<h6> 

Every exception is an instance of an exception class. 
`ValueError`, `TypeError`, `KeyError` — they’re all just classes that inherit from `BaseException`. When Python says “ValueError was raised,” it means: an instance of `ValueError` was created and thrown up the call stack.
 </h6>

In [10]:
# Prediction question — before running this, what does it print?
try:
    x = int("hello")
except ValueError as e:
    print(type(e).__name__)
    print(str(e))

ValueError
invalid literal for int() with base 10: 'hello'


<h6> <b> Try it yourself — 1 </b>

Without using any new concepts: open a Python shell and raise three different exceptions manually using raise. Try ValueError, TypeError, and KeyError. Look at what each one prints. What do they have in common?

In [ ]:
#raise ValueError('something')

#raise TypeError('expected int, got str')

raise KeyError('Revenue')

<h6>
<b> What they have in common</b>

Every traceback follows the same shape:

1, `Traceback (most recent call last)`: — the header.

2, One or more lines showing the call chain —<i> File "<stdin>", line 1, in <module></i> for a REPL.

3, A final line with the exception type and message: `ValueError: negative null_count`.

The pattern is consistent:<b> type + message</b> on the last line. That's the structure you'll read a thousand times as you debug.

<b> What's different about KeyError</b>

Notice the quotes around `revenue` in the `KeyError` output. `KeyError` wraps its argument in repr() when displaying it, so string keys appear quoted. This is a Python quirk:

In [ ]:
raise KeyError("revenue")      # displays as: KeyError: 'revenue'
print(str(KeyError("revenue"))) # prints: 'revenue' (with quotes!)

<h6>

If you want a `KeyError` with a clean message, pass a non-string or use KeyError`("revenue")` and accept the quotes. It's a known annoyance.



<h6>
<b> what should I take way.

1, Exeptions are objects</b>

They ahve tyes, attributes, methods. You can create them, inspect them, pass them around, and raise them.<i> `ValueError(...)` is calling a class constructor, same as `DatasetProfile(...)`.

<b> 2. They're organized in a class hierarchy</b>

Every exception is a subclass of `Exception` (or `BaseException`). This hierarchy matters because except can catch by parent class:

In [ ]:
try:
    x = int("hello")
except Exception as e:    # catches ValueError, TypeError, ... everything
    ...

<h6>

Catching `Exception` catches everything — usually too broad.

Catching `ValueError` catches specifically value-related errors.

You'll learn `try/except` in the next concept. For now, just know the hierarchy exists.

<b>
3. A traceback is your friend
</b>

When Python prints a traceback, it's giving you precise information:

Which exception — the class name on the last line.

What message — the text after the colon.

Where it happened — the file and line numbers in the middle.

How you got there — the call chain, bottom to top.

Reading tracebacks is a skill. You've already been doing it — every error I've debugged with you was a traceback read. The goal is to read them quickly and precisely.



<strong> **Concept 2: try / except — catching exceptions**</strong>
<h6>
<h6>

You just learned that exceptions are objects that get "thrown" when something goes wrong. Now you learn how to catch them instead of letting your program crash.

A catcher in baseball doesn’t stop the pitch — they let it come and handle it when it arrives. try/except works the same way: you let the code run, and if something is thrown, you catch it and decide what to do.

Clause|	When it runs|	Purpose|
|--------|-----------|----------|
|`try` |	Always	| The code that might fail |
| `except` SpecificError |	Only if that specific error was | raised	Handle the error |
| `else`	| Only if no exception was raised |	Code that should run only on success |
| `finally`	| Always — success or failure	| Cleanup that must happen no matter what |


In [ ]:
def load_column_count(filepath: str) -> int:
    try:
        # try: the code that might fail
        with open(filepath) as f:
            import csv
            reader = csv.DictReader(f)
            return len(reader.fieldnames)

    except FileNotFoundError:
        # except: runs only if that specific exception was raised
        print(f"File not found: {filepath}")
        return 0

    except PermissionError as e:
        # `as e` gives you the exception object — access its message
        print(f"Cannot read file: {e}")
        return 0

    else:
        # else: runs only if NO exception was raised — rarely needed
        print("File loaded cleanly")

    finally:
        # finally: ALWAYS runs — exception or not
        # use for cleanup that must happen regardless
        print("Done attempting file load")

<h6>

<u> `try`:</u>
Marks the start of the guarded block. Any exception raised inside gets caught by one of the except clauses below.

<u> `except FileNotFoundError`: </u>
Runs only if a `FileNotFoundError` was raised in the `try` block. Other exceptions (like `PermissionError`) skip this clause and look for a matching one below.

<u> `print(...)` and `return 0` </u>
Handle the error gracefully — print a message and return a default value. This is the "recover from error" pattern.

`except PermissionError as e:`

Catches only `PermissionError`.

`as e` binds the exception object to a local variable `e`.

<b> This is the key thing to notice:</b>

`except FileNotFoundError`: — catches and discards the object.

`except PermissionError as e`: — catches and gives you access to it via `e`.


`{e} in the f-string`:
`e` is the exception object. In an f-string, `{e}` calls `str(e)`, which returns the exception's message. So you print the actual error text that Python generated.

`else`:
Runs only if the `try` block finished without any exception. In this example, it would run after a successful load — but only if `return` didn't fire first.

<i> Important caveat:</i> because the `try` block ends with `return len(reader.fieldnames)`, the `else` clause never runs here. The `return` exits the function before reaching `else`.

This is a subtle point the example glosses over. The `else` clause is present to teach you the syntax, not because it's useful in this specific function. It becomes useful when the `try` block doesn't `return`.

`finally`:
Runs no matter what — success, failure, `return`, `raise`, even if an exception isn't caught. It's the "cleanup always happens" clause.

Here it just prints. In real code, `finally` is where you close files, release locks, disconnect from databases — anything that must happen regardless of outcome.

But wait — you already have `with open(...)`, which closes the file automatically. So `finally` is redundant for closing the file here. The example uses it just to demonstrate the clause.

The key insight: `except` without `as e` just catches the exception type and discards the object. `except SomeError as e` catches it AND gives you the object to inspect — message, type, traceback.

In [1]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        return -1
    else:
        return result
    finally:
        print("divide() finished")

print(divide(10, 2))
print(divide(10, 0))

divide() finished
5.0
divide() finished
-1


<h6>
<b> The critical insight </b>

Notice the order: `finally` prints BEFORE the return value reaches the caller.

This is the key behavior to internalize. `finally` runs between the return and the caller receiving the value. The return value is stored, then finally runs, then the value is delivered.

If the outer `print(...)` is what runs next, it prints after `finally` has already run. So `divide() finished` comes out first, then `5.0`.

<b> How the Clauses Interact</b>

When an exception is raised inside `try`, Python goes through this sequence:

`except` clauses are checked in order.

The first matching `except` runs. Any others are skipped.

`else` is skipped (it only runs if no exception).

`finally` runs.

- If no exception is raised:

All `except` clauses are skipped.

`else` runs.

`finally` runs.

- If an exception is raised but no `except` matches:

No `except` runs.

`else` is skipped.

`finally` runs.

Then the exception re-raises (propagates up).

Key rule: `finally` runs in every case. Even if the code inside `try` did a `return`. Even if the code inside `try` did a `raise`. Even if an `except` clause did a `return`.

This is why `finally` is the right place for cleanup — it's the one clause you can't accidentally skip.